# FinalNet on the Figshare Brain Tumor Dataset — Experiment 2 (patient-level protocol, 256×256)

EfficientNet-B0 + U-Net (scSE) + 3 deep-supervision heads + soft-mask fusion classifier (**architecture unchanged**).
Seg loss: Tversky(0.7)+BCE. Class loss: CE (label smoothing 0.1). Loss balance: learned uncertainty weights.
AdamW + cosine + EMA + grad-clip + AMP, 35 epochs, early stopping. No ASPP / TTA / Grad-CAM.

**Dataset:** Figshare brain tumor dataset (Cheng et al.), Kaggle `ashkhagan/figshare-brain-tumor-dataset`.
3,064 T1-CE slices, 233 patients, `.mat` files (`cjdata.PID / image / label / tumorMask / tumorBorder`).
Label mapping (dataset README): **1 = meningioma, 2 = glioma, 3 = pituitary**. `cvind.mat` is *not* used (we build our own patient-level folds).

**Protocol (patient-level — no image-level splitting anywhere):**
1. Stage 1 – hold-out: ~20 % of *patients* (grouped by `PID`, class-stratified as closely as possible) are set aside as the final test set.
2. Stage 2 – 5-fold stratified **group** CV on the remaining ~80 % of patients. A fold's validation part is used only for early stopping / checkpoint selection.
3. The five best-EMA fold checkpoints are ensembled (average class probabilities and segmentation probabilities, **then** threshold at 0.5).
4. The ensemble is evaluated **once** on the untouched hold-out test set → Table A (classification) and Table B (segmentation).
5. Explicit assertions verify that no patient crosses train / validation / test.

**How to use on Kaggle:** add the dataset via *Add Input*, turn Internet **On** (ImageNet weights + pip), GPU on, run all.
Set `SMOKE = True` in the config cell for a tiny end-to-end dry run first.

All outputs go to `/kaggle/working/Figshare_Experiment_Results/`.

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp

import os, re, copy, math, time, json, random, inspect, glob, warnings
from collections import Counter
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
import scipy, scipy.io as sio
from scipy import ndimage as ndi
try:
    import h5py                                   # needed only for MATLAB v7.3 (HDF5) .mat files
except ImportError:
    h5py = None
import sklearn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold          # group-aware stratification (patient-level)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             roc_curve, auc, confusion_matrix)
from tqdm.auto import tqdm
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| smp", smp.__version__, "| torch", torch.__version__,
      "| sklearn", sklearn.__version__, "| scipy", scipy.__version__, "| h5py", getattr(h5py, "__version__", None))

In [ ]:
# ---------------- CONFIG ----------------
DATASET = "FIGSHARE"   # Experiment 2: Figshare brain tumor dataset (Cheng et al.), T1-CE, 3 tumor classes
SMOKE   = False        # True = tiny quick test run (~30 patients, 2 epochs, 2 folds)

CFG = dict(
    folder="figshare-brain-tumor-dataset",             # folder name of the Kaggle input (searched under /kaggle/input)
    size=256,                                          # was 512 in the original notebook
    class_weights=False,                               # unchanged default (original D1/D3 setting)
    classes=["meningioma", "glioma", "pituitary"],     # model class index = label - 1
    label_map={1: "meningioma", 2: "glioma", 3: "pituitary"},   # from the dataset README (cjdata.label)
    expected=dict(n=3064, patients=233,                # published dataset statistics (also verified by the user)
                  per_class=dict(meningioma=708, glioma=1426, pituitary=930),
                  patients_per_class=dict(meningioma=82, glioma=89, pituitary=62)),
    paper=None,                                        # no verified published Figshare result for this protocol (see paper-comparison cell)
)
STRICT_COUNTS = True   # raise if the loaded data does not match the published counts (3,064 / 233 / per-class)

# Hyper-parameters: identical to the original notebook (epochs, patience, batch, lr, wd, ema, folds, test fraction, seed)
HP = dict(epochs=35, patience=8, batch=16, lr=2e-4, wd=1e-4, ema=0.999,
          n_folds=5, test_frac=0.2, seed=42, workers=2)
if SMOKE: HP.update(epochs=2, n_folds=2, batch=8)
SPLIT_TRIES = 20 if SMOKE else 300     # number of seeded group-split candidates screened for class balance (labels/PIDs only)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(HP["seed"])
CLASSES = CFG["classes"]; NC = len(CLASSES); SIZE = CFG["size"]
assert CLASSES == [CFG["label_map"][k] for k in sorted(CFG["label_map"])], "class order must follow label order 1,2,3"

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RES = f"{OUT}/Figshare_Experiment_Results"
for sub in ("", "checkpoints", "figures"): os.makedirs(f"{RES}/{sub}", exist_ok=True)
print("results dir:", RES, "| SIZE:", SIZE, "| seed:", HP["seed"])

In [ ]:
# ---------------- locate dataset + show its layout (check this once!) ----------------
def find_root(folder):
    for dp, dn, _ in os.walk("/kaggle/input"):
        if os.path.basename(dp) == folder:
            return dp
    return "/kaggle/input"
ROOT = find_root(CFG["folder"])
print("ROOT:", ROOT)
for dp, dn, fs in os.walk(ROOT):
    depth = dp[len(ROOT):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(dp) + f"/  ({len(fs)} files)  e.g. {fs[:2]}")

all_mat = sorted(glob.glob(f"{ROOT}/**/*.mat", recursive=True))
CVIND   = [f for f in all_mat if os.path.basename(f).lower() == "cvind.mat"]
MAT_FILES = [f for f in all_mat if os.path.basename(f).lower() != "cvind.mat"]
print(f"\n.mat files found: {len(all_mat)}  ->  sample files: {len(MAT_FILES)} | cvind.mat present: {bool(CVIND)} "
      f"(NOT used: we build our own patient-level folds)")
if not MAT_FILES:
    raise RuntimeError("No .mat sample files found under ROOT - check that the dataset was added via 'Add Input'.")

# show the label-related lines of any README shipped with the dataset (label mapping check)
docs = [f for ext in ("md", "txt") for f in glob.glob(f"{ROOT}/**/*.{ext}", recursive=True)]
for f in docs[:3]:
    try:
        txt = open(f, errors="ignore").read()
        hits = [l.strip() for l in txt.splitlines() if "label" in l.lower()][:6]
        print(f"\n[{os.path.relpath(f, ROOT)}] label-related lines:", hits if hits else "(none)")
    except Exception as e:
        print("could not read", f, e)
print("\nLabel mapping used (dataset README, cjdata.label):", CFG["label_map"])

In [ ]:
# ---------------- load (image, mask, label, PID) from the .mat files, resize to SIZE, verify ----------------
def read_cjdata(path):
    """Read one Figshare .mat file. MATLAB v5/v7 -> scipy.io.loadmat; MATLAB v7.3 (HDF5) -> h5py."""
    try:
        c = sio.loadmat(path, squeeze_me=True, struct_as_record=False)["cjdata"]
        img, msk = np.asarray(c.image), np.asarray(c.tumorMask)
        lab = np.asarray(c.label).reshape(-1)[0]
        pid = np.asarray(c.PID).reshape(-1)[0]
        fmt = "scipy"
    except NotImplementedError:                          # scipy refuses v7.3 files -> HDF5 reader
        if h5py is None: raise RuntimeError("MATLAB v7.3 file but h5py is not installed")
        with h5py.File(path, "r") as f:
            c = f["cjdata"]
            img, msk = np.array(c["image"]).T, np.array(c["tumorMask"]).T      # HDF5 stores MATLAB arrays transposed
            lab = np.array(c["label"]).reshape(-1)[0]
            pid = "".join(chr(int(v)) for v in np.array(c["PID"]).reshape(-1))  # char array stored as uint16 codes
        fmt = "h5py"
    return dict(image=img, mask=msk, label=int(lab), pid=str(pid).strip(), fmt=fmt)

def prep_image(im, size):
    """Per-slice min-max -> uint8 (the original pipeline expects 0..255 images), then resize to size x size."""
    im = im.astype(np.float32); lo, hi = float(im.min()), float(im.max())
    if hi <= lo: return None
    im8 = np.round((im - lo) / (hi - lo) * 255).astype(np.uint8)
    interp = cv2.INTER_AREA if im8.shape[0] > size else cv2.INTER_LINEAR      # AREA avoids aliasing when shrinking 512 -> 256
    return cv2.resize(im8, (size, size), interpolation=interp)

def prep_mask(m, size):
    """Binarise (>0) and resize with NEAREST so mask values stay in {0,1}."""
    return cv2.resize((np.asarray(m) > 0).astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)

def natkey(p):
    s = os.path.splitext(os.path.basename(p))[0]
    return (0, int(s), p) if s.isdigit() else (1, 0, p)

Xl, Ml, rows, excluded = [], [], [], []
for p in tqdm(sorted(MAT_FILES, key=natkey), desc="reading .mat"):
    try:
        r = read_cjdata(p); im, mk = r["image"], r["mask"]
        if im.ndim != 2 or mk.ndim != 2:           raise ValueError(f"expected 2-D image/mask, got {im.shape}/{mk.shape}")
        if im.shape != mk.shape:                   raise ValueError(f"image/mask shape mismatch {im.shape} vs {mk.shape}")
        if r["label"] not in CFG["label_map"]:     raise ValueError(f"label {r['label']} not in {sorted(CFG['label_map'])}")
        if not r["pid"]:                           raise ValueError("empty PID")
        im_s = prep_image(im, SIZE)
        if im_s is None:                           raise ValueError("constant image")
        mk_s = prep_mask(mk, SIZE)
        H, W = im.shape; px0 = int((mk > 0).sum()); px1 = int(mk_s.sum())
        rows.append(dict(file=os.path.relpath(p, ROOT), pid=r["pid"], label=r["label"], H=H, W=W, fmt=r["fmt"],
                         img_dtype=str(im.dtype), mask_nonbinary=bool(set(np.unique(mk).tolist()) - {0, 1, True, False}),
                         mask_px_orig=px0, mask_px_resized=px1,
                         area_ratio=(px1 * (H * W) / (SIZE * SIZE)) / px0 if px0 else np.nan))
        Xl.append(im_s); Ml.append(mk_s)
    except Exception as e:
        excluded.append((os.path.relpath(p, ROOT), f"{type(e).__name__}: {e}"))

META = pd.DataFrame(rows)
X = np.stack(Xl); M = np.stack(Ml); del Xl, Ml
Y = (META["label"].values - 1).astype(int)            # class index = label - 1  (meningioma, glioma, pituitary)
PID = META["pid"].values.astype(str)
HAS = META["mask_px_orig"].values > 0                  # sample has a valid (non-empty) annotated mask
FILES = META["file"].values

# ---- verification report ----
print(f"\nsample files: {len(MAT_FILES)} | loaded: {len(META)} | excluded: {len(excluded)}")
for f, why in excluded[:20]: print("  EXCLUDED", f, "->", why)
print("MAT format used:", dict(Counter(META["fmt"])))
print("original image shapes:", dict(Counter(zip(META["H"], META["W"]))), "| image dtypes:", dict(Counter(META["img_dtype"])))
print(f"resized: X{X.shape} M{M.shape} Y{Y.shape} | mask values: {np.unique(M).tolist()} | non-binary raw masks: {int(META['mask_nonbinary'].sum())}")
assert X.shape == M.shape == (len(META), SIZE, SIZE) and set(np.unique(M).tolist()) <= {0, 1}
assert (META["H"] == META["W"]).all(), "non-square image found -> resizing to a square would distort it"
# image/mask come from the SAME .mat struct, so pairing errors are impossible by construction; shapes were asserted equal per file.
print(f"empty masks in source: {int((~HAS).sum())} | tumours that vanish after resize (mask empty at {SIZE}px): {int((M.sum((1,2)) == 0).sum())} "
      f"| area ratio resized/orig min {np.nanmin(META['area_ratio']):.3f} max {np.nanmax(META['area_ratio']):.3f}")

cnt = np.bincount(Y, minlength=NC); n_pat = len(np.unique(PID))
pat_lab = META.groupby("pid")["label"].nunique()
print("class counts (images):", dict(zip(CLASSES, cnt.tolist())), "| expected:", CFG["expected"]["per_class"])
print("unique patients:", n_pat, "| expected:", CFG["expected"]["patients"],
      "| patients with >1 distinct label:", int((pat_lab > 1).sum()))
ppc = {CLASSES[c]: len(np.unique(PID[Y == c])) for c in range(NC)}
print("patients per class:", ppc, "| expected (literature):", CFG["expected"]["patients_per_class"])
ok = (len(META) == CFG["expected"]["n"] and n_pat == CFG["expected"]["patients"]
      and dict(zip(CLASSES, cnt.tolist())) == CFG["expected"]["per_class"])
print("ALL 3,064 samples / 233 patients / per-class counts accounted for:", ok)
if not ok and STRICT_COUNTS and not SMOKE:
    raise AssertionError("Loaded data does not match the published counts - inspect the exclusion list above "
                         "(set STRICT_COUNTS=False to continue anyway).")
if ppc != CFG["expected"]["patients_per_class"]:
    print("NOTE: patients-per-class differs from the literature values; the split below uses the data as loaded.")

if SMOKE:                                              # tiny patient-grouped subset: <=10 patients/class, <=4 images/patient
    rng = np.random.RandomState(HP["seed"]); keep = []
    for c in range(NC):
        for p in rng.choice(np.unique(PID[Y == c]), size=min(10, len(np.unique(PID[Y == c]))), replace=False):
            keep += np.where(PID == p)[0][:4].tolist()
    keep = np.array(sorted(keep))
    X, M, Y, HAS, PID, FILES = X[keep], M[keep], Y[keep], HAS[keep], PID[keep], FILES[keep]
    META = META.iloc[keep].reset_index(drop=True)
    print("SMOKE subset:", len(Y), "images,", len(np.unique(PID)), "patients")
print("HAS mask supervision:", int(HAS.sum()), "/", len(HAS))

# ---- visual image/mask alignment check ----
sel = np.random.RandomState(0).choice(len(Y), size=min(6, len(Y)), replace=False)
fig, ax = plt.subplots(2, len(sel), figsize=(2.6 * len(sel), 5.4), squeeze=False)
for j, i in enumerate(sel):
    ax[0, j].imshow(X[i], cmap="gray"); ax[0, j].set_title(f"{CLASSES[Y[i]]}\nPID {PID[i]}", fontsize=8)
    ax[1, j].imshow(X[i], cmap="gray")
    if M[i].any(): ax[1, j].contour(M[i], levels=[0.5], colors="r", linewidths=1)
    for a in ax[:, j]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{RES}/figures/data_alignment_check.png", dpi=120); plt.show()

In [ ]:
# ---------------- augmentation / Dataset / DataLoader (unchanged from the original notebook) ----------------
# X, M, Y, HAS are already in RAM at SIZE x SIZE (cell above). Normalisation is unchanged: (x/255 - 0.449) / 0.226.
AUG = A.Compose([A.HorizontalFlip(p=0.5),
                 A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
                 A.RandomBrightnessContrast(0.15, 0.15, p=0.5)])   # applied jointly to image + mask (mask: nearest-neighbour)

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train = idx, train
    def __len__(s): return len(s.idx)
    def __getitem__(s, i):
        j = s.idx[i]; x, m = X[j], M[j]
        if s.train:
            r = AUG(image=x, mask=m); x, m = r["image"], r["mask"]
        x = torch.from_numpy(np.ascontiguousarray(x)).float().div(255).sub(0.449).div(0.226).unsqueeze(0)
        m = torch.from_numpy(np.ascontiguousarray(m)).float().unsqueeze(0)
        return x, m, int(Y[j]), bool(HAS[j])

def loader(idx, train, bs=None):
    return DataLoader(DS(idx, train), batch_size=bs or HP["batch"], shuffle=train, drop_last=train,
                      num_workers=HP["workers"], pin_memory=True, persistent_workers=HP["workers"] > 0)

# quick check: one augmented sample keeps a binary mask and the right shapes
_x, _m, _y, _h = DS(np.arange(len(Y)), True)[0]
assert _x.shape == (1, SIZE, SIZE) and _m.shape == (1, SIZE, SIZE) and set(torch.unique(_m).tolist()) <= {0.0, 1.0}
print("dataset ok | sample:", tuple(_x.shape), tuple(_m.shape), "label", _y)

In [ ]:
# ---------------- PATIENT-LEVEL splitting: hold-out test + 5-fold stratified GROUP CV ----------------
# Groups = PID. Only StratifiedGroupKFold is used (no StratifiedShuffleSplit / StratifiedKFold). Patient separation has priority
# over class balance. To get the most balanced partition we screen SPLIT_TRIES seeded StratifiedGroupKFold candidates using ONLY
# labels / PIDs / sizes (never images or model performance) and keep the best one. The chosen seeds are saved for reproducibility.
all_idx = np.arange(len(Y))
pid_u = np.unique(PID)
pid_label = {p: int(np.bincount(Y[PID == p], minlength=NC).argmax()) for p in pid_u}     # majority label per patient
multi = [p for p in pid_u if len(np.unique(Y[PID == p])) > 1]
print(f"patients: {len(pid_u)} | patients with more than one label: {len(multi)} {multi[:5]}")
Y_STRAT = np.array([pid_label[p] for p in PID])          # patient-level label broadcast to images (== Y if every patient has 1 label)
print("Y_STRAT == Y for all images:", bool((Y_STRAT == Y).all()))

def best_group_folds(y, groups, n_splits, seed, tries, first_only=False):
    g = np.bincount(y, minlength=NC) / len(y); best = None
    for t in range(tries):
        rs = seed + t
        try:
            folds = [te for _, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=rs)
                     .split(np.zeros(len(y)), y, groups)]
        except ValueError:
            continue
        use = folds[:1] if first_only else folds
        sc = float(np.mean([np.abs(np.bincount(y[f], minlength=NC) / max(len(f), 1) - g).sum()
                            + 2 * abs(len(f) / len(y) - 1 / n_splits) for f in use]))
        if best is None or sc < best[0]: best = (sc, rs, folds)
    if best is None: raise RuntimeError("could not build a group split (too few patients?)")
    return best[2], best[1], best[0]

# Stage 1: hold-out test patients (~test_frac of patients)
HO_SPLITS = int(round(1 / HP["test_frac"]))
ho_folds, ho_seed, ho_score = best_group_folds(Y_STRAT, PID, HO_SPLITS, HP["seed"], SPLIT_TRIES, first_only=True)
te_idx = np.sort(ho_folds[0]); tv_idx = np.setdiff1d(all_idx, te_idx)

# Stage 2: K-fold stratified group CV on the train/val patients
cv_rel, cv_seed, cv_score = best_group_folds(Y_STRAT[tv_idx], PID[tv_idx], HP["n_folds"], HP["seed"], SPLIT_TRIES)
FOLDS = []
for va_rel in cv_rel:
    va = tv_idx[np.sort(va_rel)]; tr = np.setdiff1d(tv_idx, va); FOLDS.append((tr, va))
print(f"hold-out: StratifiedGroupKFold(n_splits={HO_SPLITS}) random_state={ho_seed} balance-score={ho_score:.4f}")
print(f"CV      : StratifiedGroupKFold(n_splits={HP['n_folds']}) random_state={cv_seed} balance-score={cv_score:.4f}")

# ---------------- explicit leakage assertions ----------------
ps = lambda idx: set(PID[idx])
P_te, P_tv = ps(te_idx), ps(tv_idx)
assert len(P_tv & P_te) == 0, "LEAK: a patient is in both train/val and test"
assert len(set(tv_idx) & set(te_idx)) == 0 and len(tv_idx) + len(te_idx) == len(Y)
assert P_tv | P_te == set(pid_u)
val_sets = []
for k, (tr, va) in enumerate(FOLDS):
    Ptr, Pva = ps(tr), ps(va)
    assert len(Ptr & Pva) == 0, f"LEAK fold {k}: train PID intersect validation PID != 0"
    assert len(Ptr & P_te) == 0, f"LEAK fold {k}: train PID intersect test PID != 0"
    assert len(Pva & P_te) == 0, f"LEAK fold {k}: validation PID intersect test PID != 0"
    assert len(set(tr) & set(va)) == 0 and len(tr) + len(va) == len(tv_idx)
    assert Ptr | Pva == P_tv
    val_sets.append(Pva)
for i in range(len(val_sets)):
    for j in range(i + 1, len(val_sets)):
        assert len(val_sets[i] & val_sets[j]) == 0, f"LEAK: patient in validation of folds {i} and {j}"
assert set().union(*val_sets) == P_tv and sum(len(v) for v in val_sets) == len(P_tv)
print("PATIENT-LEVEL LEAKAGE CHECKS PASSED: train∩val = train∩test = val∩test = 0 patients; every train/val patient is in exactly one validation fold.")

# ---------------- report patients / images / class distribution of every split ----------------
def split_stats(name, idx):
    pids = ps(idx); ci = np.bincount(Y[idx], minlength=NC); cp = np.bincount([pid_label[p] for p in pids], minlength=NC)
    d = dict(split=name, n_patients=len(pids), n_images=len(idx))
    d.update({f"img_{c}": int(n) for c, n in zip(CLASSES, ci)}); d.update({f"pat_{c}": int(n) for c, n in zip(CLASSES, cp)})
    return d
srows = [split_stats("ALL", all_idx), split_stats("train+val (hold-in)", tv_idx), split_stats("TEST (hold-out)", te_idx)]
for k, (tr, va) in enumerate(FOLDS):
    srows += [split_stats(f"fold{k} train", tr), split_stats(f"fold{k} val", va)]
SPLIT_SUMMARY = pd.DataFrame(srows)
print(); print(SPLIT_SUMMARY.to_string(index=False))
for r in srows:
    miss = [c for c in CLASSES if r[f"img_{c}"] == 0]
    if miss: print(f"WARNING: split '{r['split']}' has no images of class(es) {miss} (AUC for them will be NaN)")

# ---------------- save split information (seed + patient IDs per split) ----------------
fold_of_img = np.full(len(Y), -1)
for k, (tr, va) in enumerate(FOLDS): fold_of_img[va] = k
part = np.where(np.isin(all_idx, te_idx), "test", "trainval")
pd.DataFrame(dict(sample_index=all_idx, file=FILES, pid=PID, label=Y + 1, cls=[CLASSES[c] for c in Y],
                  partition=part, cv_val_fold=fold_of_img)).to_csv(f"{RES}/image_split_assignment.csv", index=False)
pd.DataFrame([dict(pid=p, cls=CLASSES[pid_label[p]], n_images=int((PID == p).sum()),
                   partition="test" if p in P_te else "trainval",
                   cv_val_fold=int(fold_of_img[np.where(PID == p)[0][0]])) for p in pid_u]
             ).sort_values(["partition", "cv_val_fold", "pid"]).to_csv(f"{RES}/patient_split_assignment.csv", index=False)
SPLIT_SUMMARY.to_csv(f"{RES}/split_summary.csv", index=False)
json.dump(dict(random_seed=HP["seed"], dataset=DATASET, n_images=int(len(Y)), n_patients=int(len(pid_u)),
               holdout=dict(n_splits=HO_SPLITS, selected_random_state=int(ho_seed), balance_score=ho_score,
                            n_candidates_screened=SPLIT_TRIES, test_pids=sorted(P_te)),
               cv=dict(n_folds=HP["n_folds"], selected_random_state=int(cv_seed), balance_score=cv_score,
                       folds=[dict(fold=k, train_pids=sorted(ps(tr)), val_pids=sorted(ps(va))) for k, (tr, va) in enumerate(FOLDS)]),
               note="reproduce with StratifiedGroupKFold(n_splits, shuffle=True, random_state=selected_random_state) on "
                    "(labels broadcast per patient, groups=PID); see image_split_assignment.csv for the exact per-image assignment"),
          open(f"{RES}/split_info.json", "w"), indent=1)
print("saved split files to", RES)

In [ ]:
# ---------------- FinalNet ----------------
class FinalNet(nn.Module):
    def __init__(self, num_classes, pretrained=True, enc="tu-efficientnet_b0"):
        super().__init__()
        dch = (256, 128, 64, 32, 16)
        base = smp.Unet(enc, encoder_weights=True if pretrained else None, in_channels=3, classes=1,
                        decoder_channels=dch, decoder_attention_type="scse")
        self.encoder, self.decoder, self.seg_head = base.encoder, base.decoder, base.segmentation_head
        self.aux_ids = (1, 2, 3)                                   # 3 auxiliary 1x1 heads
        self.aux_heads = nn.ModuleList([nn.Conv2d(dch[i], 1, 1) for i in self.aux_ids])
        self._f = {}
        for i in self.aux_ids:
            self.decoder.blocks[i].register_forward_hook(self._mk(i))
        ps = inspect.signature(self.decoder.forward).parameters
        self._star = any(p.kind == p.VAR_POSITIONAL for p in ps.values())   # smp-version compatibility
        c32, c16 = self.encoder.out_channels[-1], self.encoder.out_channels[-2]
        self.head = nn.Sequential(nn.Linear(c32 * 2 + c16, 256), nn.ReLU(True), nn.Dropout(0.3),
                                  nn.Linear(256, num_classes))

    def _mk(self, i):
        def hook(mod, inp, out): self._f[i] = out
        return hook

    def forward(self, x):
        feats = self.encoder(x)
        d = self.decoder(*feats) if self._star else self.decoder(feats)
        seg = self.seg_head(d)
        aux = [h(self._f[i]) for h, i in zip(self.aux_heads, self.aux_ids)]
        m = torch.sigmoid(seg.float())                              # soft mask: no threshold, no detach
        f32, f16 = feats[-1].float(), feats[-2].float()             # stride 32 and stride 16
        def wpool(f):
            mm = F.interpolate(m, size=f.shape[-2:], mode="bilinear", align_corners=False)
            return (f * mm).sum((2, 3)) / (mm.sum((2, 3)) + 1e-6)
        z = torch.cat([f32.mean((2, 3)), wpool(f32), wpool(f16)], 1)
        return dict(seg=seg, aux=aux, cls=self.head(z))

# ---------------- losses ----------------
def tversky(logit, t, alpha=0.7, eps=1.0):
    p = torch.sigmoid(logit.float())
    tp = (p * t).sum(); fn = ((1 - p) * t).sum(); fp = (p * (1 - t)).sum()
    return 1 - (tp + eps) / (tp + alpha * fn + (1 - alpha) * fp + eps)

def seg_loss(out, tgt, valid):
    if valid.sum() == 0: return None
    tot = 0; outs = [out["seg"]] + out["aux"]
    for o in outs:
        o, t = o[valid], tgt[valid]
        if o.shape[-2:] != t.shape[-2:]:
            o = F.interpolate(o.float(), size=t.shape[-2:], mode="bilinear", align_corners=False)
        tot = tot + tversky(o, t) + F.binary_cross_entropy_with_logits(o.float(), t)
    return tot / len(outs)

class Uncertainty(nn.Module):                                      # learned task weights (Kendall et al.)
    def __init__(s): super().__init__(); s.s = nn.Parameter(torch.zeros(2))
    def forward(s, ls, lc):
        t = torch.exp(-s.s[1]) * lc + s.s[1]
        if ls is not None: t = t + torch.exp(-s.s[0]) * ls + s.s[0]
        return t

class EMA:
    def __init__(s, model, decay):
        s.m = FinalNet(NC, pretrained=False).to(dev); s.m.load_state_dict(model.state_dict()); s.m.eval()
        for p in s.m.parameters(): p.requires_grad_(False)
        s.d, s.n = decay, 0
    @torch.no_grad()
    def update(s, model):
        s.n += 1; d = min(s.d, (1 + s.n) / (10 + s.n))
        for e, p in zip(s.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(p.detach(), alpha=1 - d)
            else: e.copy_(p)

In [ ]:
# ---------------- sanity check at SIZE x SIZE: output shapes, 3 classes, one train step, CUDA memory ----------------
_m = FinalNet(NC, pretrained=False).to(dev); _m.train()
_x = torch.randn(HP["batch"], 3, SIZE, SIZE, device=dev)
_t = (torch.rand(HP["batch"], 1, SIZE, SIZE, device=dev) > 0.8).float(); _y = torch.randint(0, NC, (HP["batch"],), device=dev)
if dev.type == "cuda": torch.cuda.reset_peak_memory_stats()
with torch.autocast(dev.type, enabled=dev.type == "cuda"):
    _o = _m(_x)
print("main seg:", tuple(_o["seg"].shape), "| aux:", [tuple(a.shape) for a in _o["aux"]], "| cls:", tuple(_o["cls"].shape))
assert _o["seg"].shape == (HP["batch"], 1, SIZE, SIZE), "main segmentation output must be SIZE x SIZE"
assert len(_o["aux"]) == 3 and all(a.shape[-1] == a.shape[-2] and a.shape[-1] <= SIZE for a in _o["aux"])
assert _o["cls"].shape == (HP["batch"], NC) and NC == 3, "classification output must have exactly 3 classes"
_ls = seg_loss(_o, _t, torch.ones(HP["batch"], dtype=torch.bool, device=dev))      # aux outputs are resized to SIZE inside seg_loss
_loss = Uncertainty().to(dev)(_ls, nn.functional.cross_entropy(_o["cls"].float(), _y)); _loss.backward()
assert torch.isfinite(_loss)
if dev.type == "cuda": print(f"peak CUDA memory for one train step (batch {HP['batch']}, {SIZE}x{SIZE}): {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print("architecture check passed | trainable params: %.2fM" % (sum(p.numel() for p in _m.parameters() if p.requires_grad) / 1e6))
del _m, _x, _t, _y, _o, _ls, _loss
if dev.type == "cuda": torch.cuda.empty_cache()

In [ ]:
# ---------------- metrics (numpy): validation metrics used in training + full Table A / Table B ----------------
SEG_THR = 0.5                                   # ONE threshold everywhere (validation, test, ensemble), applied after probability averaging
HD_PENALTY = float(np.hypot(SIZE, SIZE))        # image diagonal in pixels @ SIZExSIZE; used only for the *penalised* HD95 variant

def cls_metrics(y, prob):                        # unchanged: used for validation / early stopping
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return dict(acc=accuracy_score(y, pred), prec=p, rec=r, f1=f)

def seg_stats(prob, gt, thr=SEG_THR):
    """Per-image binary segmentation statistics. prob (N,H,W) probabilities, gt (N,H,W) {0,1}.
    Conventions for degenerate cases (nothing is dropped):
      both pred and GT empty -> Dice = IoU = precision = recall = 1 (correct 'nothing');
      GT non-empty, pred empty (failed detection) -> Dice = IoU = recall = 0, precision = 0;
      GT empty, pred non-empty (false alarm)      -> Dice = IoU = precision = 0, recall = 1 (vacuous);
      specificity is 1 when there are no negative pixels."""
    pred = prob > thr; g = gt > 0.5
    tp = (pred & g).sum((1, 2)).astype(np.float64); fp = (pred & ~g).sum((1, 2)).astype(np.float64)
    fn = (~pred & g).sum((1, 2)).astype(np.float64); tn = (~pred & ~g).sum((1, 2)).astype(np.float64)
    sp, sg = pred.sum((1, 2)).astype(np.float64), g.sum((1, 2)).astype(np.float64)
    both_empty = (sp == 0) & (sg == 0)
    return dict(
        dice=np.where(both_empty, 1.0, 2 * tp / np.maximum(sp + sg, 1)),
        iou=np.where(both_empty, 1.0, tp / np.maximum(sp + sg - tp, 1)),
        precision=np.where(sp > 0, tp / np.maximum(sp, 1), np.where(sg == 0, 1.0, 0.0)),
        recall=np.where(sg > 0, tp / np.maximum(sg, 1), 1.0),
        specificity=np.where((tn + fp) > 0, tn / np.maximum(tn + fp, 1), 1.0),
        tp=tp, fp=fp, fn=fn, tn=tn, pred_px=sp, gt_px=sg)

def dice_imgs(mprob, gt, has=None):
    """Per-image Dice (macro over images) used for validation. Includes every image that has mask supervision
    (`has`); images with an empty GT are NOT dropped (empty/empty -> 1, false alarm -> 0)."""
    d = seg_stats(mprob, gt)["dice"]
    return d if has is None else d[np.asarray(has, bool)]

_CROSS = ndi.generate_binary_structure(2, 1)
def _surface(m): return m & ~ndi.binary_erosion(m, structure=_CROSS, border_value=0)

def hd95_single(pred, gt):
    """95th-percentile Hausdorff distance in PIXELS (MedPy-style: max of the two directed 95th percentiles of
    surface-to-surface distances). Both empty -> 0. Exactly one empty -> NaN (undefined; counted & reported)."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any():  return float("nan")
    sp, sg = _surface(pred), _surface(gt)
    d_to_gt = ndi.distance_transform_edt(~sg)[sp]; d_to_pred = ndi.distance_transform_edt(~sp)[sg]
    return float(max(np.percentile(d_to_gt, 95), np.percentile(d_to_pred, 95)))

def full_cls_report(y, P, classes=CLASSES):
    """Table A. Macro precision/recall/F1 over ALL classes (zero_division=0); one-vs-rest AUC per class
    (NaN if a class is absent or is the only class in the split) and macro-AUC = mean over the classes where it is defined."""
    y = np.asarray(y); K = len(classes); pred = P.argmax(1); labs = list(range(K))
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=labs, average="macro", zero_division=0)
    pc, rc, fc, sc = precision_recall_fscore_support(y, pred, labels=labs, average=None, zero_division=0)
    aucs = []
    for c in labs:
        yb = (y == c)
        aucs.append(float(roc_auc_score(yb, P[:, c])) if 0 < yb.sum() < len(yb) else float("nan"))
    valid = [a for a in aucs if not np.isnan(a)]
    macro_auc = float(np.mean(valid)) if valid else float("nan")
    if len(valid) == K and K > 2:                # cross-check against scikit-learn's multiclass OvR macro AUC
        Pn = P / P.sum(1, keepdims=True)
        assert abs(macro_auc - roc_auc_score(y, Pn, multi_class="ovr", average="macro", labels=labs)) < 1e-6
    cm = confusion_matrix(y, pred, labels=labs)
    overall = dict(n=int(len(y)), accuracy=float(accuracy_score(y, pred)), macro_precision=float(p), macro_recall=float(r),
                   macro_f1=float(f), macro_auc_ovr=macro_auc, n_classes_with_auc=len(valid))
    per_class = pd.DataFrame(dict(cls=classes, precision=pc, recall=rc, f1=fc, auc_ovr=aucs, support=sc.astype(int)))
    return overall, per_class, cm

def full_seg_report(prob, gt, y, classes=CLASSES, thr=SEG_THR, show_progress=True):
    """Table B + per-class Dice on ALL images (nothing excluded). Dice/IoU/precision/recall/specificity are reported both as
    the mean of per-image values (macro over images) and globally pooled over all pixels. HD95 is per image, in pixels."""
    s = seg_stats(prob, gt, thr); N = len(y); pred = prob > thr; g = gt > 0.5
    it = tqdm(range(N), desc="HD95", leave=False) if show_progress else range(N)
    hd = np.array([hd95_single(pred[i], g[i]) for i in it])
    hd_def = hd[~np.isnan(hd)]
    pen = np.where(np.isnan(hd), HD_PENALTY, hd)
    def pooled(sel):
        TP, FP, FN, TN = (s[k][sel].sum() for k in ("tp", "fp", "fn", "tn"))
        dn = 2 * TP + FP + FN
        return dict(dice=float(2 * TP / dn) if dn else 1.0, iou=float(TP / (TP + FP + FN)) if (TP + FP + FN) else 1.0,
                    precision=float(TP / (TP + FP)) if (TP + FP) else (1.0 if FN == 0 else 0.0),
                    recall=float(TP / (TP + FN)) if (TP + FN) else 1.0,
                    specificity=float(TN / (TN + FP)) if (TN + FP) else 1.0)
    allsel = np.ones(N, bool); G = pooled(allsel)
    summary = dict(
        threshold=thr, n_images=int(N), resolution=f"{SIZE}x{SIZE}",
        dice_per_image_mean=float(s["dice"].mean()), dice_per_image_std=float(s["dice"].std()), dice_per_image_median=float(np.median(s["dice"])),
        dice_global_pooled=G["dice"],
        iou_per_image_mean=float(s["iou"].mean()), iou_global_pooled=G["iou"],
        precision_per_image_mean=float(s["precision"].mean()), precision_global_pooled=G["precision"],
        recall_per_image_mean=float(s["recall"].mean()), recall_global_pooled=G["recall"],
        specificity_per_image_mean=float(s["specificity"].mean()), specificity_global_pooled=G["specificity"],
        hd95_px_mean_defined=float(hd_def.mean()) if len(hd_def) else float("nan"),
        hd95_px_median_defined=float(np.median(hd_def)) if len(hd_def) else float("nan"),
        hd95_px_mean_penalised=float(pen.mean()), hd95_penalty_px=HD_PENALTY,
        n_hd95_defined=int(len(hd_def)), n_hd95_undefined=int(np.isnan(hd).sum()),
        n_empty_pred=int((s["pred_px"] == 0).sum()), n_empty_gt=int((s["gt_px"] == 0).sum()), n_dice_zero=int((s["dice"] == 0).sum()))
    rowsc = []
    for c, name in enumerate(classes):
        sel = (np.asarray(y) == c); pc = pooled(sel) if sel.any() else dict(dice=np.nan, iou=np.nan)
        hc = hd[sel]; hcd = hc[~np.isnan(hc)]
        rowsc.append(dict(cls=name, n=int(sel.sum()),
                          dice_per_image_mean=float(s["dice"][sel].mean()) if sel.any() else np.nan,
                          dice_per_image_std=float(s["dice"][sel].std()) if sel.any() else np.nan,
                          dice_global_pooled=pc["dice"], iou_per_image_mean=float(s["iou"][sel].mean()) if sel.any() else np.nan,
                          hd95_px_mean_defined=float(hcd.mean()) if len(hcd) else np.nan, n_hd95_undefined=int(np.isnan(hc).sum()),
                          n_empty_pred=int((s["pred_px"][sel] == 0).sum())))
    per_img = pd.DataFrame(dict(dice=s["dice"], iou=s["iou"], precision=s["precision"], recall=s["recall"],
                                specificity=s["specificity"], hd95_px=hd, pred_px=s["pred_px"].astype(int), gt_px=s["gt_px"].astype(int)))
    return summary, pd.DataFrame(rowsc), per_img

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, float): return None if np.isnan(o) else o
    if isinstance(o, np.ndarray): return jsonable(o.tolist())
    return o

In [ ]:
# ---------------- prediction / validation evaluation (torch) ----------------
@torch.no_grad()
def predict(model, dl, keep_masks=False, with_dice=True):
    """Returns class probabilities (N,NC), per-image Dice (or None) and, if keep_masks, mask probabilities (N,H,W) float32.
    Order == the order of the indices the loader was built from (shuffle=False)."""
    model.eval(); P, Ds, MP = [], [], []
    for x, m, y, has in dl:
        x = x.to(dev, non_blocking=True).expand(-1, 3, -1, -1)
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            out = model(x)
        assert out["seg"].shape[-2:] == x.shape[-2:] and out["cls"].shape[-1] == NC
        P.append(F.softmax(out["cls"].float(), 1).cpu())
        sm = torch.sigmoid(out["seg"].float())[:, 0].cpu()
        if with_dice: Ds.append(dice_imgs(sm.numpy(), m[:, 0].numpy(), has.numpy()))
        if keep_masks: MP.append(sm)
    P = torch.cat(P).numpy()
    return P, (np.concatenate(Ds) if with_dice else None), (torch.cat(MP).numpy() if keep_masks else None)

def evaluate(model, idx, keep_masks=False):
    """Used on VALIDATION folds only (early stopping / checkpoint selection)."""
    P, Ds, MP = predict(model, loader(idx, False, HP["batch"] * 2), keep_masks)
    r = cls_metrics(Y[idx], P); r["dice"] = float(Ds.mean()) if len(Ds) else float("nan")
    return r, P, MP

In [ ]:
# ---------------- one fold (training logic unchanged; adds history logging + checkpoint path in the results folder) ----------------
def train_fold(fold, tr_idx, va_idx):
    seed_all(HP["seed"] + fold)
    model = FinalNet(NC).to(dev); ema = EMA(model, HP["ema"]); unc = Uncertainty().to(dev)
    cw = None
    if CFG["class_weights"]:
        c = np.bincount(Y[tr_idx], minlength=NC); cw = torch.tensor(len(tr_idx) / (NC * np.maximum(c, 1)), dtype=torch.float32, device=dev)
    ce = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    opt = torch.optim.AdamW([dict(params=model.parameters(), weight_decay=HP["wd"]),
                             dict(params=unc.parameters(), weight_decay=0.0, lr=1e-3)], lr=HP["lr"])
    dl = loader(tr_idx, True); steps = HP["epochs"] * len(dl); warm = len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    best, bad, path = -1, 0, f"{RES}/checkpoints/fold{fold}_best_ema.pt"
    hist, best_ep = [], -1
    if dev.type == "cuda": torch.cuda.reset_peak_memory_stats()
    for ep in range(HP["epochs"]):
        model.train(); t0 = time.time(); tl = 0
        for x, m, y, has in dl:
            x = x.to(dev, non_blocking=True).expand(-1, 3, -1, -1); m = m.to(dev); y = y.to(dev); has = has.to(dev)
            with torch.autocast(dev.type, enabled=dev.type == "cuda"):
                out = model(x)
            ls = seg_loss(out, m, has); lc = ce(out["cls"].float(), y)
            loss = unc(ls, lc)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model); tl += loss.item()
        r, _, _ = evaluate(ema.m, va_idx)                       # VALIDATION fold only - the test set is never touched here
        score = r["f1"] + (0 if math.isnan(r["dice"]) else r["dice"])
        print(f"fold{fold} ep{ep+1:02d} loss {tl/len(dl):.4f} | val acc {r['acc']:.4f} f1 {r['f1']:.4f} "
              f"dice {r['dice']:.4f} | w=({unc.s[0].item():.2f},{unc.s[1].item():.2f}) | {time.time()-t0:.0f}s")
        hist.append(dict(fold=fold, epoch=ep + 1, train_loss=tl / len(dl), val_acc=r["acc"], val_prec=r["prec"], val_rec=r["rec"],
                         val_f1=r["f1"], val_dice=r["dice"], val_score=score, lr=opt.param_groups[0]["lr"],
                         w_seg=unc.s[0].item(), w_cls=unc.s[1].item(), epoch_time_s=time.time() - t0))
        if score > best: best, bad, best_ep = score, 0, ep + 1; torch.save(ema.m.state_dict(), path)
        else:
            bad += 1
            if bad >= HP["patience"]: print("early stop"); break
    if dev.type == "cuda": print(f"fold{fold} peak CUDA memory: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
    return path, hist, dict(best_epoch=best_ep, best_val_score=best)

In [ ]:
# ---------------- run: K folds (validation used for early stopping / checkpoint selection ONLY; no test evaluation here) ----------------
print(f"train+val={len(tv_idx)} imgs / {len(P_tv)} patients | test={len(te_idx)} imgs / {len(P_te)} patients (untouched until the ensemble cell)")
paths, histories, fold_rows = [], [], []
for k, (tr, va) in enumerate(FOLDS):
    path, hist, info = train_fold(k, tr, va); paths.append(path); histories += hist
    mdl = FinalNet(NC, pretrained=False).to(dev); mdl.load_state_dict(torch.load(path, map_location=dev))
    r, _, _ = evaluate(mdl, va)                                   # best EMA checkpoint on its own VALIDATION fold
    fold_rows.append(dict(fold=k, n_train_images=len(tr), n_val_images=len(va), n_val_patients=len(ps(va)), **info,
                          **{f"val_{a}": b for a, b in r.items()}))
    print(f"==> fold {k} VAL (best EMA ckpt, epoch {info['best_epoch']}): " + "  ".join(f"{n} {v:.4f}" for n, v in r.items()))
    del mdl; torch.cuda.empty_cache()
    pd.DataFrame(histories).to_csv(f"{RES}/training_history.csv", index=False)      # saved after every fold (crash-safe)

HIST = pd.DataFrame(histories); FOLD_VAL = pd.DataFrame(fold_rows)
HIST.to_csv(f"{RES}/training_history.csv", index=False); HIST.to_json(f"{RES}/training_history.json", orient="records", indent=1)
FOLD_VAL.to_csv(f"{RES}/fold_validation_metrics.csv", index=False)
print(f"\n===== per-fold VALIDATION metrics (best EMA checkpoint; mean +- std over {len(paths)} folds) =====")
for key in ["val_acc", "val_prec", "val_rec", "val_f1", "val_dice"]:
    print(f"{key:9s} {np.nanmean(FOLD_VAL[key]):.4f} +- {np.nanstd(FOLD_VAL[key]):.4f}")
print("(validation folds were used for checkpoint selection, so these are optimistic; the unbiased estimate is the test-set ensemble below)")

# final ensemble checkpoint = the five best-EMA fold state-dicts in a single file (+ metadata)
torch.save(dict(state_dicts=[torch.load(p, map_location="cpu") for p in paths], classes=CLASSES, size=SIZE, seg_threshold=SEG_THR,
                hp=HP, seed=HP["seed"], arch="FinalNet: EfficientNet-B0 U-Net + scSE + 3 aux heads + soft-mask fusion classifier",
                ensemble="average softmax probabilities and sigmoid mask probabilities over folds, then threshold"),
           f"{RES}/checkpoints/final_ensemble.pt")
print("saved:", [os.path.basename(p) for p in paths], "+ final_ensemble.pt")

In [ ]:
# ---------------- 5-fold ENSEMBLE: evaluated ONCE on the untouched hold-out test set ----------------
# One pass: average class probabilities and mask probabilities over the 5 fold models, THEN threshold at SEG_THR.
te_dl = loader(te_idx, False, HP["batch"] * 2)
P_sum, M_sum = 0, 0
for path in paths:
    mdl = FinalNet(NC, pretrained=False).to(dev); mdl.load_state_dict(torch.load(path, map_location=dev))
    P, _, MP = predict(mdl, te_dl, keep_masks=True, with_dice=False)
    P_sum = P_sum + P; M_sum = M_sum + MP; del mdl, MP; torch.cuda.empty_cache()
P_ens = P_sum / len(paths); M_ens = M_sum / len(paths)
del P_sum, M_sum
Y_te, M_te = Y[te_idx], M[te_idx]
assert P_ens.shape == (len(te_idx), NC) and np.allclose(P_ens.sum(1), 1, atol=1e-4)
assert M_ens.shape == (len(te_idx), SIZE, SIZE) == M_te.shape

# keep the raw ensemble outputs so every number below can be recomputed without touching the test set again
np.savez_compressed(f"{RES}/test_ensemble_outputs.npz", test_index=te_idx, pid=PID[te_idx], y=Y_te, class_prob=P_ens.astype(np.float32),
                    mask_prob=M_ens.astype(np.float16), threshold=SEG_THR)

# ---------------- Table A: classification ----------------
cls_overall, cls_per_class, CM = full_cls_report(Y_te, P_ens)
tableA = pd.DataFrame([dict(metric=k, value=v) for k, v in [("Accuracy", cls_overall["accuracy"]),
        ("Macro Precision", cls_overall["macro_precision"]), ("Macro Recall", cls_overall["macro_recall"]),
        ("Macro F1", cls_overall["macro_f1"]), ("Macro ROC-AUC (OvR)", cls_overall["macro_auc_ovr"])]])
print(f"===== TABLE A - classification, 5-fold ensemble on the hold-out test set ({len(te_idx)} images / {len(P_te)} patients) =====")
print(tableA.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nPer-class (one-vs-rest):"); print(cls_per_class.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nConfusion matrix (rows=true, cols=pred):", CLASSES); print(CM)

# ---------------- Table B: segmentation ----------------
seg_sum, seg_per_class, seg_img = full_seg_report(M_ens, M_te, Y_te)
tableB = pd.DataFrame([dict(metric=k, value=v) for k, v in [
    ("Dice (mean per-image, PRIMARY)", seg_sum["dice_per_image_mean"]), ("Dice (global, pooled pixels)", seg_sum["dice_global_pooled"]),
    ("IoU (mean per-image)", seg_sum["iou_per_image_mean"]), ("IoU (global)", seg_sum["iou_global_pooled"]),
    ("Pixel Precision (mean per-image)", seg_sum["precision_per_image_mean"]), ("Pixel Precision (global)", seg_sum["precision_global_pooled"]),
    ("Pixel Recall/Sensitivity (mean per-image)", seg_sum["recall_per_image_mean"]), ("Pixel Recall/Sensitivity (global)", seg_sum["recall_global_pooled"]),
    ("Specificity (mean per-image)", seg_sum["specificity_per_image_mean"]), ("Specificity (global)", seg_sum["specificity_global_pooled"]),
    ("HD95 px (mean, images where defined)", seg_sum["hd95_px_mean_defined"]), ("HD95 px (median, defined)", seg_sum["hd95_px_median_defined"]),
    (f"HD95 px (mean, undefined cases = {HD_PENALTY:.1f}px penalty)", seg_sum["hd95_px_mean_penalised"])]])
print(f"\n===== TABLE B - tumour segmentation, same ensemble & test set, threshold {SEG_THR}, {SIZE}x{SIZE} =====")
print(tableB.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nempty predictions: {seg_sum['n_empty_pred']} | empty GT masks: {seg_sum['n_empty_gt']} | Dice==0 images: {seg_sum['n_dice_zero']} | "
      f"HD95 defined/undefined: {seg_sum['n_hd95_defined']}/{seg_sum['n_hd95_undefined']}  (none of these images were dropped from the means above, except undefined HD95 in the 'defined' variant)")
print("\nPer-class Dice (test images grouped by GROUND-TRUTH class):"); print(seg_per_class.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nDice definition: PRIMARY = mean of per-image Dice over ALL test images (macro over images, every test image included); "
      "'global' = Dice of pooled pixels over the whole test set.")

# ---------------- save tables / json / csv ----------------
per_class_all = cls_per_class.merge(seg_per_class.rename(columns={"n": "n_images_seg"}), on="cls")
tableA.to_csv(f"{RES}/classification_results.csv", index=False)
tableB.to_csv(f"{RES}/segmentation_results.csv", index=False)
per_class_all.to_csv(f"{RES}/per_class_metrics.csv", index=False)
pimg = pd.DataFrame(dict(sample_index=te_idx, file=FILES[te_idx], pid=PID[te_idx], y_true=Y_te, y_pred=P_ens.argmax(1),
                         **{f"p_{c}": P_ens[:, i] for i, c in enumerate(CLASSES)})).join(seg_img)
pimg.to_csv(f"{RES}/test_per_image_results.csv", index=False)
common = dict(dataset=DATASET, seed=HP["seed"], n_test_images=int(len(te_idx)), n_test_patients=len(P_te), ensemble_members=len(paths),
              test_evaluated="once, after probability averaging over the five best-EMA fold models")
json.dump(jsonable(dict(**common, overall=cls_overall, per_class=cls_per_class.to_dict("records"),
                        confusion_matrix=dict(labels=CLASSES, rows_true_cols_pred=CM.tolist()),
                        fold_validation_mean_std={k: [float(np.nanmean(FOLD_VAL[k])), float(np.nanstd(FOLD_VAL[k]))] for k in ["val_acc", "val_prec", "val_rec", "val_f1", "val_dice"]})),
          open(f"{RES}/classification_metrics.json", "w"), indent=2)
json.dump(jsonable(dict(**common, overall=seg_sum, per_class=seg_per_class.to_dict("records"),
                        conventions="Dice primary = mean per-image Dice over all test images; empty/empty -> 1; HD95 in pixels at "
                                    f"{SIZE}x{SIZE}; undefined HD95 (exactly one mask empty) reported separately and as {HD_PENALTY:.1f}px-penalised mean")),
          open(f"{RES}/segmentation_metrics.json", "w"), indent=2)
json.dump(jsonable(dict(hp=HP, size=SIZE, seg_threshold=SEG_THR, classes=CLASSES, label_map=CFG["label_map"], seed=HP["seed"],
                        split_random_states=dict(holdout=int(ho_seed), cv=int(cv_seed)), torch=torch.__version__, smp=smp.__version__,
                        sklearn=sklearn.__version__, scipy=scipy.__version__, cuda=dev.type == "cuda")),
          open(f"{RES}/run_config.json", "w"), indent=2)
print("\nsaved tables / json / csv to", RES)

In [ ]:
# ---------------- figures: confusion matrix, ROC, per-class Dice, training curves, representative predictions ----------------
def plot_confusion(cm, classes, path):
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.2, 4.4)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{cm[i, j]}\n({cmn[i, j]:.1%})", ha="center", va="center", fontsize=9, color="white" if cmn[i, j] > .5 else "black")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=20); ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix - hold-out test, 5-fold ensemble")
    plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def plot_roc(y, P, classes, path):
    fig, ax = plt.subplots(figsize=(5.6, 5)); grid = np.linspace(0, 1, 201); tprs = []
    for c, name in enumerate(classes):
        yb = (y == c)
        if 0 < yb.sum() < len(yb):                                      # classes absent from the split are skipped safely
            fpr, tpr, _ = roc_curve(yb, P[:, c]); a = auc(fpr, tpr); ax.plot(fpr, tpr, label=f"{name} (AUC {a:.3f})")
            tprs.append(np.interp(grid, fpr, tpr))
        else: print(f"ROC: class '{name}' skipped (absent or only class in this split)")
    if tprs:
        mt = np.mean(tprs, 0); ax.plot(grid, mt, "k--", lw=2, label=f"macro-average (AUC {auc(grid, mt):.3f})")
    ax.plot([0, 1], [0, 1], ":", color="gray"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("One-vs-rest ROC - hold-out test, 5-fold ensemble"); ax.legend(loc="lower right"); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def plot_class_dice(seg_pc, path):
    fig, ax = plt.subplots(figsize=(5.2, 4)); x = np.arange(len(seg_pc))
    ax.bar(x - .18, seg_pc["dice_per_image_mean"], .36, yerr=seg_pc["dice_per_image_std"], capsize=3, label="mean per-image Dice (+-SD)")
    ax.bar(x + .18, seg_pc["dice_global_pooled"], .36, label="global pooled Dice")
    ax.set_xticks(x); ax.set_xticklabels([f"{c}\n(n={n})" for c, n in zip(seg_pc["cls"], seg_pc["n"])]); ax.set_ylim(0, 1.05)
    ax.set_ylabel("Dice"); ax.set_title("Per-class Dice - hold-out test"); ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def plot_history(hist, path):
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
    for f, h in hist.groupby("fold"):
        for a, col, t in zip(ax, ["train_loss", "val_acc", "val_f1", "val_dice"], ["train loss", "val accuracy", "val macro-F1", "val Dice (per-image)"]):
            a.plot(h["epoch"], h[col], label=f"fold {f}"); a.set_title(t); a.set_xlabel("epoch")
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def plot_representative(Xte, Mte, Pte, y, dice, classes, path, thr=SEG_THR):
    """Per class: worst / median / best test image by per-image Dice (qualitative only; picked after evaluation)."""
    picks = []
    for c in range(len(classes)):
        ix = np.where(y == c)[0]
        if len(ix):
            o = ix[np.argsort(dice[ix])]; picks += [(c, "worst", o[0]), (c, "median", o[len(o) // 2]), (c, "best", o[-1])]
    fig, ax = plt.subplots(len(picks), 4, figsize=(10, 2.5 * len(picks)), squeeze=False)
    for r, (c, tag, i) in enumerate(picks):
        gt, pr = Mte[i] > 0.5, Pte[i] > thr
        ax[r, 0].imshow(Xte[i], cmap="gray"); ax[r, 0].set_title(f"{classes[c]} - {tag}", fontsize=8)
        ax[r, 1].imshow(gt, cmap="gray"); ax[r, 1].set_title("ground truth", fontsize=8)
        ax[r, 2].imshow(pr, cmap="gray"); ax[r, 2].set_title(f"ensemble pred (Dice {dice[i]:.3f})", fontsize=8)
        ax[r, 3].imshow(Xte[i], cmap="gray"); ax[r, 3].set_title("overlay: GT green / pred red", fontsize=8)
        if gt.any(): ax[r, 3].contour(gt, levels=[.5], colors="lime", linewidths=1)
        if pr.any(): ax[r, 3].contour(pr, levels=[.5], colors="red", linewidths=1)
        for a in ax[r]: a.axis("off")
    plt.tight_layout(); plt.savefig(path, dpi=130); plt.show()

FIG = f"{RES}/figures"
plot_confusion(CM, CLASSES, f"{FIG}/confusion_matrix_test.png")
plot_roc(Y_te, P_ens, CLASSES, f"{FIG}/roc_curves_test.png")
plot_class_dice(seg_per_class, f"{FIG}/per_class_dice_test.png")
plot_history(HIST, f"{FIG}/training_history.png")
plot_representative(X[te_idx], M_te, M_ens, Y_te, seg_img["dice"].values, CLASSES, f"{FIG}/representative_segmentations.png")
print("figures saved to", FIG)

In [ ]:
# ---------------- paper comparison (only with VERIFIED published Figshare numbers) ----------------
# The original notebook compared with the paper's Table 13 for datasets D1/D2/D3 only. No published Figshare result with a verifiable
# protocol is available to us, so NO paper number is used or carried over from another dataset. To enable a comparison, fill PAPER_REF
# with numbers you have verified in the source, including how that source split the data.
PAPER_REF = None
# PAPER_REF = dict(source="<citation / table>", split_protocol="<how the paper split train/test (image-level? patient-level?)>",
#                  acc=None, f1=None, dice=None)
if PAPER_REF is None:
    print("Paper comparison skipped: no verified published Figshare results/protocol supplied (nothing is fabricated or reused from D1-D3).")
else:
    cmp = pd.DataFrame([dict(row="Original paper (as published)", acc=PAPER_REF["acc"], f1=PAPER_REF["f1"], dice=PAPER_REF["dice"]),
                        dict(row="Ours: 5-fold ensemble, patient-level hold-out", acc=cls_overall["accuracy"], f1=cls_overall["macro_f1"],
                             dice=seg_sum["dice_per_image_mean"])])
    print(cmp.to_string(index=False)); print("\nPaper source:", PAPER_REF["source"]); print("Paper split protocol:", PAPER_REF["split_protocol"])
    print("Ours: patient-level (PID) hold-out ~20% + patient-level 5-fold CV, test used once, 256x256, Dice = mean per-image over all test images.")
    print("Differences in splitting / resolution / Dice definition mean the two rows are NOT directly comparable unless the protocols match.")

**Notes / conventions**
- **Splitting:** everything is grouped by `PID` (StratifiedGroupKFold). No image-level split exists anywhere. Class balance is as good as grouping allows (see `split_summary.csv`); patient separation always has priority. The seeds and exact patient IDs of every split are in `split_info.json`, `patient_split_assignment.csv` and `image_split_assignment.csv`.
- **Model selection:** checkpoint selection and early stopping use only the validation fold (macro-F1 + per-image Dice, as in the original notebook). The hold-out test set is touched **once**, by the 5-fold ensemble, after averaging probabilities.
- **Segmentation:** one binary tumour mask, threshold 0.5 applied after averaging the five models' sigmoid probabilities. **Primary Dice = mean of per-image Dice over *all* test images** (macro over images; empty/empty = 1, missed or false-alarm = 0). Global pooled Dice/IoU/precision/recall/specificity are reported alongside. HD95 is in pixels at 256×256 (MedPy-style); images where exactly one of prediction/GT is empty have undefined HD95 — they are counted (`n_hd95_undefined`) and also included in a penalised mean (image diagonal ≈ 362 px).
- **Per-class Dice** groups test images by their ground-truth tumour class.
- **Classification:** macro precision/recall/F1 over the 3 classes; one-vs-rest AUC per class, macro-AUC = mean of the per-class AUCs (NaN-safe if a class is missing from a split; cross-checked against scikit-learn's `roc_auc_score(multi_class="ovr")` when all classes are present).
- **Pre-processing deviations (necessary for `.mat` data):** per-slice min-max → uint8, then the original normalisation `(x/255−0.449)/0.226`; `INTER_AREA` when shrinking 512→256 (linear for already-256 images); masks `INTER_NEAREST`.
- If you hit CUDA OOM, lower `HP["batch"]` to 8. GPU runs are not bit-wise deterministic even with fixed seeds.